In [ ]:
import torch
from src.extractors.visual.visual_feature_extractor import VisualFeatureExtractor
from src.common.constants import OPENFACE_BIN

device = "cuda" if torch.cuda.is_available() else "cpu"

extractor = VisualFeatureExtractor(
    openface_bin_path=OPENFACE_BIN,
    device=device
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionPooling(nn.Module):
    """Standalone Attention Pooling module.
    Computes a learned weighted sum across the sequence/temporal dimension.
    """
    def __init__(self, input_dim: int, attention_dim: int = 128):
        super().__init__()
        self.attention_net = nn.Sequential(
            nn.Linear(input_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1)
        )

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, input_dim)
        Returns:
            pooled: Tensor of shape (batch_size, input_dim)
            weights: Tensor of shape (batch_size, seq_len, 1)
        """
        scores = self.attention_net(x)
        weights = F.softmax(scores, dim=1)
        pooled = torch.sum(x * weights, dim=1)
        return pooled, weights

In [ ]:
import torch
import torch.nn as nn
class VisualBiLSTMEncoder(nn.Module):
    """Bidirectional LSTM encoder for visual features (dim=58) 
    with optional Attention Pooling (may use it or remove it).
    """
    def __init__(
        self, 
        input_dim: int = 58, 
        hidden_dim: int = 128, 
        num_layers: int = 2, 
        dropout: float = 0.3,
        pooling_mode: str = "attention",  # Options: "attention", "mean", "last"
        attention_module: nn.Module = None
    ):
        super().__init__()
        self.pooling_mode = pooling_mode
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        lstm_output_dim = hidden_dim * 2  # Bidirectional doubles the dimension
        
        # Configure or attach the pooling module if requested
        if pooling_mode == "attention":
            if attention_module is not None:
                self.pool = attention_module
            else:
                self.pool = AttentionPooling(input_dim=lstm_output_dim, attention_dim=hidden_dim)
        else:
            self.pool = None

        # Projection/fusion alignment head
        self.projection = nn.Sequential(
            nn.Linear(lstm_output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor, return_weights: bool = False):
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, 58)
            return_weights: If True and using attention, returns weights alongside features
        """
        lstm_out, _ = self.lstm(x)
        
        weights = None
        if self.pooling_mode == "attention":
            pooled, weights = self.pool(lstm_out)
        elif self.pooling_mode == "mean":
            pooled = torch.mean(lstm_out, dim=1)
        elif self.pooling_mode == "last":
            pooled = lstm_out[:, -1, :]
        else:
            raise ValueError(f"Unknown pooling mode: {self.pooling_mode}")
        
        fusion_features = self.projection(pooled)
        
        if return_weights and self.pooling_mode == "attention":
            return fusion_features, weights
        return fusion_features

In [ ]:
print(f"Confirmed Extractor Feature Dimension: {extractor.feature_dim}")  # Output: 58

# Initialize Bi-LSTM Encoder
visual_encoder = VisualBiLSTMEncoder(
    input_dim=extractor.feature_dim,  # Matches 58 dimensions[cite: 1]
    hidden_dim=128,
    num_layers=2,
    pooling_mode="attention"  # Set to "mean" or "last" if you wish to remove attention
).to(device)
visual_encoder.eval()

# Run full pipeline on a sample video
video_path = "path/to/your/sample_video.mp4"

with torch.no_grad():
    # Extract frame-level features -> shape: (1, target_t, 58)[cite: 1]
    visual_features = extractor.extract_features(video_path, target_t=90).to(device)
    
    # Pass through Bi-LSTM encoder
    fusion_vector, attn_weights = visual_encoder(visual_features, return_weights=True)
    
    print("-" * 40)
    print("Pipeline Execution Successful!")
    print("Extracted Video Tensor Shape:      ", visual_features.shape)
    print("Multimodal Fusion-Ready Vector Shape:", fusion_vector.shape)
    print("Attention Weights Shape:             ", attn_weights.shape)